# Silver Layer — Cleaned & Validated Data

Silver layer transformations:
- **PII Masking**: Email, phone, name hashed with SHA-256
- **Deduplication**: Window-based row_number dedup
- **Data Quality**: `dq_is_valid` flag, `dq_reason` column

In [ ]:
%pip install pyspark==3.5.3

In [ ]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_silver") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://dataops-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataops-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataops-secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark {spark.version} ready")

In [ ]:
BASE = "s3a://delta-lake/silver"
tables = ["customers", "products", "transactions"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    df.printSchema()
    df.show(5, truncate=False)

In [ ]:
# PII masking verification
df = spark.read.format("delta").load(f"{BASE}/customers")
print("Email looks hashed:")
df.select("email_hashed").show(5, truncate=False)
print("Phone looks hashed:")
df.select("phone_hashed").show(5, truncate=False)

In [ ]:
# Data quality distribution
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{t.upper()} DQ stats:")
    df.groupBy("dq_is_valid").count().show()

In [ ]:
# Compare bronze vs silver row counts
BRONZE = "s3a://delta-lake/bronze"
for t in tables:
    b = spark.read.format("delta").load(f"{BRONZE}/{t}").count()
    s = spark.read.format("delta").load(f"{BASE}/{t}").count()
    print(f"{t}: bronze={b}  silver={s}  diff={b-s}")

In [ ]:
spark.stop()